In [1]:
# We need these libraries to clean and process data and perform calculations
import pandas as pd
import numpy as np
import re


Part 1 - load our data 
remove unnessary chars in dtaa

In [2]:
# Read all of our Excel files
meta = pd.read_excel("metaClean43Brightspace.xlsx")
sales = pd.read_excel("sales.xlsx")


In [3]:
# Converiting the Excel files to CSV files for easier handling in the future
meta.to_csv("meta.csv", index=False)
sales.to_csv("sales.csv", index=False)


In [4]:
# Printing the first 10 rows of the sales dataset to check if it has been read correctly
print(sales.head(10))

   year     release_date                                  title  \
0  2000      January 1st                           Bakha Satang   
1  2001     January 12th                              Antitrust   
2  2000     January 28th                               Santitos   
3  2002  2002 (Wide) by                     Frank McKlusky C.I.   
4  2002     January 25th                     A Walk to Remember   
5  2002        June 21st                                Zig Zag   
6  2002         May 10th                         TakhtÃƒÂ© siah   
7  2009      January 2nd  Angry Monk: Reflections on Tibet, The   
8  2002         June 7th                       30 Years to Life   
9  2008      January 2nd             The Killing of John Lennon   

               genre  international_box_office  domestic_box_office  \
0              Drama                   76576.0                  NaN   
1  Thriller/Suspense                 6900000.0           10965209.0   
2                NaN                       NaN   

In [5]:
print(meta.head(10))

                                                 url                   title  \
0  https://www.metacritic.com/movie/!women-art-re...   !Women Art Revolution   
1  https://www.metacritic.com/movie/10-cloverfiel...     10 Cloverfield Lane   
2  https://www.metacritic.com/movie/10-items-or-less        10 Items or Less   
3          https://www.metacritic.com/movie/10-years                10 Years   
4  https://www.metacritic.com/movie/100-bloody-acres        100 Bloody Acres   
5       https://www.metacritic.com/movie/100-streets             100 Streets   
6  https://www.metacritic.com/movie/1000-times-go...  1,000 Times Good Night   
7          https://www.metacritic.com/movie/10000-bc               10,000 BC   
8          https://www.metacritic.com/movie/10000-km               10,000 km   
9        https://www.metacritic.com/movie/1001-grams              1001 Grams   

                     studio       rating  runtime  \
0       Hotwire Productions  | Not Rated     83.0   
1        Para

In [6]:
# We only take the columns we need MOVIE + SCORE columns 
meta_cols = meta[[
    "url",         
    "title",      
    "studio",      # issue 8: needed so we can merge studio name variants + build the movie-pk file
    "runtime",     
    "rating",      
    "RelDate",     
    "metascore",   
    "userscore",   
    "genre",       
    "cast",        
    "director",    
]].copy()
meta_cols.head()

,url,title,studio,runtime,rating,RelDate,metascore,userscore,genre,cast,director
0,https://www.metacritic.com/movie/!women-art-re...,!Women Art Revolution,Hotwire Productions,83.0,| Not Rated,2011-06-01,70,NaN,Documentary,NaN,Lynn Hershman-Leeson
1,https://www.metacritic.com/movie/10-cloverfiel...,10 Cloverfield Lane,Paramount Pictures,104.0,| PG-13,2016-03-11,76,7.7,"Action,Sci-Fi,Drama,Mystery,Thriller,Horror","John Gallagher Jr.,John Goodman,Mary Elizabeth...",Dan Trachtenberg
2,https://www.metacritic.com/movie/10-items-or-less,10 Items or Less,Click Star,82.0,| R,2006-12-01,54,5.8,"Drama,Comedy,Romance","Jonah Hill,Morgan Freeman,Paz Vega",Brad Silberling
3,https://www.metacritic.com/movie/10-years,10 Years,Anchor Bay Entertainment,100.0,| R,2012-09-14,61,6.9,"Drama,Comedy,Romance","Channing Tatum,Chris Pratt,Jenna Dewan",Jamie Linden
4,https://www.metacritic.com/movie/100-bloody-acres,100 Bloody Acres,Music Box Films,91.0,| Not Rated,2013-06-28,63,7.5,"Horror,Comedy",NaN,Cameron Cairnes


In [7]:
# SALES columns (from sales)
sales_cols = sales[[
    "title",                    
    "production_budget",        
    "theatre_count",            
    "international_box_office",  
    "domestic_box_office",      
]].copy()
sales_cols.head()
print(sales.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30612 entries, 0 to 30611
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   year                      30612 non-null  int64  
 1   release_date              30612 non-null  object 
 2   title                     30604 non-null  object 
 3   genre                     28908 non-null  object 
 4   international_box_office  21575 non-null  float64
 5   domestic_box_office       11884 non-null  float64
 6   worldwide_box_office      21575 non-null  float64
 7   production_budget         4480 non-null   float64
 8   Unnamed: 8                0 non-null      float64
 9   opening_weekend           10929 non-null  float64
 10  theatre_count             10963 non-null  float64
 11  avg run per theatre       10952 non-null  float64
 12  runtime                   24559 non-null  float64
 13  keywords                  12517 non-null  object 
 14  creati

In [8]:
# Clean meta_cols that have unncessary characters and whitespace like | and sapces.
meta_cols["rating"] = meta_cols["rating"].str.replace("|", "", regex=False).str.strip()
meta_cols["title"] = meta_cols["title"].str.strip()
meta_cols.head()

,url,title,studio,runtime,rating,RelDate,metascore,userscore,genre,cast,director
0,https://www.metacritic.com/movie/!women-art-re...,!Women Art Revolution,Hotwire Productions,83.0,Not Rated,2011-06-01,70,NaN,Documentary,NaN,Lynn Hershman-Leeson
1,https://www.metacritic.com/movie/10-cloverfiel...,10 Cloverfield Lane,Paramount Pictures,104.0,PG-13,2016-03-11,76,7.7,"Action,Sci-Fi,Drama,Mystery,Thriller,Horror","John Gallagher Jr.,John Goodman,Mary Elizabeth...",Dan Trachtenberg
2,https://www.metacritic.com/movie/10-items-or-less,10 Items or Less,Click Star,82.0,R,2006-12-01,54,5.8,"Drama,Comedy,Romance","Jonah Hill,Morgan Freeman,Paz Vega",Brad Silberling
3,https://www.metacritic.com/movie/10-years,10 Years,Anchor Bay Entertainment,100.0,R,2012-09-14,61,6.9,"Drama,Comedy,Romance","Channing Tatum,Chris Pratt,Jenna Dewan",Jamie Linden
4,https://www.metacritic.com/movie/100-bloody-acres,100 Bloody Acres,Music Box Films,91.0,Not Rated,2013-06-28,63,7.5,"Horror,Comedy",NaN,Cameron Cairnes


In [9]:
# Check for empty values (NULL) in the datasets

nan_report = pd.DataFrame({
    "nan_count": meta_cols.isna().sum(),
    "nan_percent": (meta_cols.isna().mean() * 100).round(1)
})
nan_report

,nan_count,nan_percent
url,0,0.0
title,0,0.0
studio,350,3.1
runtime,255,2.2
rating,1067,9.4
RelDate,0,0.0
metascore,0,0.0
userscore,2105,18.5
genre,20,0.2
cast,3702,32.6


In [10]:
# Nan finder for sales_cols
nan_report = pd.DataFrame({
    "nan_count": sales_cols.isna().sum(),
    "nan_percent": (sales_cols.isna().mean() * 100).round(1)
})
nan_report

,nan_count,nan_percent
title,8,0.0
production_budget,26132,85.4
theatre_count,19649,64.2
international_box_office,9037,29.5
domestic_box_office,18728,61.2


Part Two 
This part we will using data processing techniques to prepare our data and put them all into one master file
Issues that we have to fix: 
1 - metascore (0–100) and userscore (0–10) on different scales (so we will make it 100, so both can be int.) - done
2 - meta and sales have no shared ID (Metacritic vs The-Numbers urls) - done
3 - TODO: 85% of budgets missing ( average based on genre and studio) - use the studios as well, they should be the same, perferably done after 8 
4 - 693 duplicate titles in sales - done
5 - 550k individual reviews, but box office is per-movie - done
6 - Raw revenue just tracks budget - done (added ROI; budget rounded to 2 decimals as currency)
7 - movieID should be a unique ID based on release date, same thing for every primary key - done
8 - studios should be merged into one, like 20th fox and fox home entertainment should be one studio -  then keep the manual info 
10 - TODO: another movie-pk csv file that only includes MovieID, Title, studio, Runtime, Rating, Release date, Production budget - update the movie-pk file 
11 - TODO: change the production budget to currancy 

In [11]:
#1: metascore is 0-100 but userscore is 0-10, so we will put userscore on the same 0-100 scale so the gap is meaningful.

meta_cols["userscore100"] = meta_cols["userscore"] * 10 #our new converted userscore is userscore100
print(meta_cols["userscore100"].head(20))

0      NaN
1     77.0
2     58.0
3     69.0
4     75.0
5     61.0
6     68.0
7     46.0
8     74.0
9      NaN
10    54.0
11    51.0
12    67.0
13     NaN
14    75.0
15     NaN
16    74.0
17     NaN
18    58.0
19    66.0
Name: userscore100, dtype: float64


In [12]:
#2: meta and sales have no shared ID, so we will normalise the title on both tables and use it as the join key.

def norm_title(t):
    t = str(t).lower().strip()
    t = re.sub(r'[^a-z0-9 ]', '', t)     # drop punctuation / accented characters
    t = re.sub(r'\s+', ' ', t).strip()   # collapse repeated whitespace
    return t

# the .map() runs our norm_title function on every value in the title column and returns a new column so we get a cleaned up title on both tables. Same function on both
meta_cols['title_norm'] = meta_cols['title'].map(norm_title)
sales['title_norm']     = sales['title'].map(norm_title)


# quick check: how many meta titles now find a match in sales
matched = meta_cols['title_norm'].isin(set(sales['title_norm'])).sum()
print(f"{matched} of {len(meta_cols)} meta titles matched a sales title")
meta_cols[['title', 'title_norm']].head(20)

8964 of 11364 meta titles matched a sales title


,title,title_norm
0,!Women Art Revolution,women art revolution
1,10 Cloverfield Lane,10 cloverfield lane
2,10 Items or Less,10 items or less
3,10 Years,10 years
4,100 Bloody Acres,100 bloody acres
5,100 Streets,100 streets
6,"1,000 Times Good Night",1000 times good night
7,"10,000 BC",10000 bc
8,"10,000 km",10000 km
9,1001 Grams,1001 grams


In [13]:
#3: ~85% of the sales rows have no production_budget. Dropping them would throw away most of
#   the data.
#   We fill each missing budget with the budget of films in the SAME GENRE, because films of a
#   similar genre tend to have comparable budgets, so the genre value is a fair estimate.
#   We use the MEDIAN rather than a plain average: budgets are heavily right-skewed (a few
#   blockbusters inflate the mean), and the median is robust to those outliers.
#TODO: production budget must be a currancy, and have less than 3 decimal places 


# Median budget per genre, learned only from the rows that already have a real budget (> 0)
genre_budget = sales.loc[sales['production_budget'] > 0].groupby('genre')['production_budget'].median()

# Global median as a fallback for rows whose genre is missing or has no budgeted film
global_budget = sales.loc[sales['production_budget'] > 0, 'production_budget'].median()

# Fill: genre median first, then the global median for anything still missing
sales['production_budget'] = (sales['production_budget']
    .fillna(sales['genre'].map(genre_budget))
    .fillna(global_budget))

print("production_budget still missing:", sales['production_budget'].isna().sum())
sales[['title', 'genre', 'production_budget']].head()

production_budget still missing: 0


,title,genre,production_budget
0,Bakha Satang,Drama,12000000.0
1,Antitrust,Thriller/Suspense,30000000.0
2,Santitos,NaN,20000000.0
3,Frank McKlusky C.I.,NaN,20000000.0
4,A Walk to Remember,Drama,11000000.0


In [14]:
#4: sales has 629 duplicate titles (the same film listed more than once). If we leave them,
#   one movie would match several sales rows and get double-counted in the merge.
#   We keep ONE row per normalised title: sort by worldwide_box_office and keep the largest
#   (the main theatrical release), then drop the rest.

#This could also be done with a if loop, why not if loop??
before = len(sales)
sales_dedup = (sales
    .sort_values('worldwide_box_office', ascending=False)   # biggest release first
    .drop_duplicates(subset='title_norm', keep='first'))    # keep it, drop the duplicates

print(f"removed {before - len(sales_dedup)} duplicate rows, {len(sales_dedup)} unique titles left")

removed 693 duplicate rows, 29919 unique titles left


In [15]:
#8: the same company is written many different ways (for example "Twentieth Century Fox Film
#   Corporation", "Fox Searchlight Pictures", "Fox Home Entertainment"). We merge these
#   variants into ONE canonical studio name so each real studio is counted once.

# First tidy the raw text: drop leading/trailing spaces
meta_cols["studio"] = meta_cols["studio"].str.strip()

# Map keyword -> canonical parent studio. If a studio name contains any of the keywords,
# it belongs to that parent. Anything not matched keeps its own (cleaned) name.
studio_groups = {
    "20th Century Fox":      ["fox"],
    "Warner Bros.":          ["warner", "new line"],
    "Universal Pictures":    ["universal", "focus features"],
    "Sony Pictures":         ["sony", "columbia", "tristar", "screen gems"],
    "Walt Disney":           ["disney", "buena vista", "pixar", "marvel", "lucasfilm"],
    "Paramount Pictures":    ["paramount"],
    "Lionsgate":             ["lionsgate", "lions gate", "summit"],
    "Metro-Goldwyn-Mayer":   ["metro-goldwyn", "mgm"],
    "The Weinstein Company": ["weinstein", "miramax"],
}

def canon_studio(name):
    low = str(name).lower()
    for canonical, keywords in studio_groups.items():
        if any(k in low for k in keywords):   # any variant keyword found -> use the parent name
            return canonical
    return name                               # not a known major -> leave as-is

before = meta_cols["studio"].nunique()
meta_cols["studio"] = meta_cols["studio"].map(canon_studio)
print(f"studios reduced from {before} to {meta_cols['studio'].nunique()} unique names")
meta_cols["studio"].value_counts().head(10)

studios reduced from 1118 to 1031 unique names


Sony Pictures         797
Warner Bros.          556
Universal Pictures    511
20th Century Fox      491
Lionsgate             444
IFC Films             408
Paramount Pictures    351
Walt Disney           311
Netflix               301
Magnolia Pictures     268
Name: studio, dtype: int64

In [16]:
#6: raw revenue mostly just tracks the budget (big budget -> big revenue), so on its own it is a
#   poor measure of success. We build ONE master table and add ROI, which measures revenue
#   RELATIVE to budget and is therefore comparable across both big and small films.

# Keep only the deduped sales columns we actually need for the join
sales_match = sales_dedup[['title_norm', 'production_budget', 'worldwide_box_office']]

# Merge everything into one movie-level master table
master = (meta_cols
    .merge(sales_match, on='title_norm', how='left'))  # issue 2: sales, joined on normalised title

# The two variables our research question needs
master['gap'] = master['userscore100'] - master['metascore']                 # issue 1: both on 0-100
master['roi'] = master['worldwide_box_office'] / master['production_budget']  # issue 6: revenue vs budget

print('master:', master.shape,
      '| modeling-ready (gap + roi):', (master['gap'].notna() & master['roi'].notna()).sum())
master.head()

master: (11364, 17) | modeling-ready (gap + roi): 5549


,url,title,studio,runtime,rating,RelDate,metascore,userscore,genre,cast,director,userscore100,title_norm,production_budget,worldwide_box_office,gap,roi
0,https://www.metacritic.com/movie/!women-art-re...,!Women Art Revolution,Hotwire Productions,83.0,Not Rated,2011-06-01,70,NaN,Documentary,NaN,Lynn Hershman-Leeson,NaN,women art revolution,1000000.0,NaN,NaN,NaN
1,https://www.metacritic.com/movie/10-cloverfiel...,10 Cloverfield Lane,Paramount Pictures,104.0,PG-13,2016-03-11,76,7.7,"Action,Sci-Fi,Drama,Mystery,Thriller,Horror","John Gallagher Jr.,John Goodman,Mary Elizabeth...",Dan Trachtenberg,77.0,10 cloverfield lane,15000000.0,108286422.0,1.0,7.219095
2,https://www.metacritic.com/movie/10-items-or-less,10 Items or Less,Click Star,82.0,R,2006-12-01,54,5.8,"Drama,Comedy,Romance","Jonah Hill,Morgan Freeman,Paz Vega",Brad Silberling,58.0,10 items or less,NaN,NaN,4.0,NaN
3,https://www.metacritic.com/movie/10-years,10 Years,Anchor Bay Entertainment,100.0,R,2012-09-14,61,6.9,"Drama,Comedy,Romance","Channing Tatum,Chris Pratt,Jenna Dewan",Jamie Linden,69.0,10 years,12000000.0,987640.0,8.0,0.082303
4,https://www.metacritic.com/movie/100-bloody-acres,100 Bloody Acres,Music Box Films,91.0,Not Rated,2013-06-28,63,7.5,"Horror,Comedy",NaN,Cameron Cairnes,75.0,100 bloody acres,10500000.0,NaN,12.0,NaN


In [ ]:
#7: every table needs a proper PRIMARY KEY. We give each movie a unique MovieID built from
#   the release date order: sort the movies by RelDate, then number them 1..N.

# Sort by release date (oldest first) and rebuild a clean 0..N-1 index
master = master.sort_values(["RelDate","title"]).reset_index(drop=True)

# Zero-padded sequential id, e.g. M00001, M00002 ... -> guaranteed unique, ordered by release
master.insert(0, 'movieid', range(1, len(master) + 1))

# Give the aggregated review tables the SAME MovieID (their natural key is url) so they can
# reference a movie by its primary key instead of the long url.
url_to_id = master.set_index("url")["MovieID"]          # lookup: url -> MovieID

print("MovieID unique?", master["MovieID"].is_unique, "| rows:", len(master))
master[["MovieID", "title", "RelDate"]].head()

TypeError: DataFrame.insert() missing 1 required positional argument: 'value'

In [ ]:
#10: build a small "movie primary key" table with only the core identifying columns and
#    save it as its own CSV.

# Pick the required columns from master and give them clean, readable names
movie_pk = master[[
    "MovieID",
    "title",
    "studio",
    "runtime",
    "rating",
    "RelDate",
    "production_budget",
]].copy()
movie_pk.columns = [
    "MovieID", "Title", "Studio", "Runtime", "Rating", "ReleaseDate", "ProductionBudget"
]

# issue 6 TODO: production budget is money, so keep it to 2 decimal places (currency format)
movie_pk["ProductionBudget"] = movie_pk["ProductionBudget"].round(2)

# Save the primary-key table
movie_pk.to_csv("movie_pk.csv", index=False)
print("saved movie_pk.csv:", movie_pk.shape)
movie_pk.head()

saved movie_pk.csv: (11364, 7)


,MovieID,Title,Studio,Runtime,Rating,ReleaseDate,ProductionBudget
0,M00001,Fantasia 2000,Walt Disney,74.0,G,2000-01-01,NaN
1,M00002,Lupin III: The Castle of Cagliostro,Eleven Arts,100.0,PG-13,2000-01-01,NaN
2,M00003,Next Friday,Warner Bros.,98.0,R,2000-01-12,9500000.0
3,M00004,My Dog Skip,Warner Bros.,95.0,PG,2000-01-12,7000000.0
4,M00005,Supernova,United Artists,90.0,R,2000-01-14,60000000.0


In [ ]:
# Save the combined table
master.to_csv("movieslookup.csv", index=False)